<a href="https://colab.research.google.com/github/Evgeniya-Ryasnova/generative-mocap/blob/master/generative_mocap_reproduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
Tesla T4


In [2]:
!pwd
!ls

/content
sample_data


In [3]:
!git clone https://github.com/Evgeniya-Ryasnova/generative-mocap.git


Cloning into 'generative-mocap'...
remote: Enumerating objects: 237, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 237 (delta 6), reused 0 (delta 0), pack-reused 220 (from 4)
Receiving objects: 100% (237/237), 1.38 GiB | 31.58 MiB/s, done.
Resolving deltas: 100% (27/27), done.
Updating files: 100% (184/184), done.


In [4]:
%cd generative-mocap

/content/generative-mocap


In [5]:
!ls

compare_test_data.py			 LICENSE
compare_train_data.py			 plot_results.py
data					 previous_papers.py
datasets.py				 Readme.md
dimensionless_walking_speed_analysis.py  reproduction_colab_Ryasnova.ipynb
dynamical_consistency.py		 Results
environment.yml				 train_cgans.py
Figures					 transformation.py
generative_mocap_reproduction.ipynb	 utils.py


In [6]:
!ls data

data_1.pickle  data_2.pickle


In [7]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Tesla T4


In [8]:
!pip install spm1d ezc3d statsmodels

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 77.9 MB/s eta 0:00:00


In [9]:
!head -50 train_cgans.py

# -*- coding: utf-8 -*-
"""

Author:   Metin Bicer
email:    m.bicer19@imperial.ac.uk

"""
import torch.nn as nn
import torch
from torch.autograd import Variable
from datasets import mocapDataLoader
from transformation import HeightChannelScale, labelScale
import numpy as np
import itertools
import pickle as pkl
import time
import os
from itertools import product
import random


### DEFAULTS ###

# Headers for the markers, grfs and ik angles
MARKER_NAMES = np.array(['L_IAS', 'L_IPS', 'R_IPS', 'R_IAS', 'R_FTC',
                         'R_FLE', 'R_FME', 'R_FAX', 'R_TTC', 'R_FAL',
                         'R_TAM', 'R_FCC', 'R_FM1', 'R_FM2', 'R_FM5'])
# the followings are headers only for the x comp of f, m and cop
# because of the dataframe structure
GRF_NAMES = np.array(['force','point','moment'])
IK_NAMES = np.array(['pelvis_tilt', 'pelvis_list', 'pelvis_rotation',
                     'hip_flexion_r', 'hip_adduction_r', 'hip_rotation_r',
                     'knee_angle_r', 'ankle_ang

In [10]:
!python train_cgans.py

/content/generative-mocap/utils.py:486: SyntaxWarning: invalid escape sequence '\%'
  for i in [4, 5, 9]: axs[i].set_xlabel('Gait Cycle [\%]', fontsize=25)
/content/generative-mocap/utils.py:598: SyntaxWarning: invalid escape sequence '\%'
  for ax in axs[2:]: ax.set_xlabel('Gait Cycle [\%]', fontsize=25)


In [11]:
!grep -n "if __name__" train_cgans.py

In [12]:
!tail -120 train_cgans.py

    value_cols = ['marker_gc', 'grf_3d_gc', 'ik_gc']
    names_cols = ['marker_names', 'grf_names_3d', 'ik_names']
    # labels in label_cols
    label_col_contd = ['walking_speed']
    label_col_discr = None
    # file
    data_df_file = ['data/data_1.pickle', 'data/data_2.pickle']
    excluded_subjects = [2014001, 2014003, 2015042]
    # hyperparams
    z_dim = 20
    hidden_dim = 512
    n_epochs = 3000
    lr = 0.002
    batch_size = 128
    display_step = 250
    n_samples = 10
    # define ages at which n_samples of data will be generated
    label_contd_lims = [[0.16, 2.41, 0.02]]
    label_discr_lims = None
    model_name = 'wscgan'
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_cgan(marker_names, grf_names, ik_names, value_cols,
               names_cols, label_col_contd, label_col_discr,
               data_df_file, excluded_subjects, z_dim, hidden_dim,
               n_epochs, lr, batch_size, display_step, n_samples,
               label_

In [13]:
!sed -n '650,980p' train_cgans.py

    back_transform_label, transform_labels = transformations[2:]
    # generate and save synthetic data
    # define conditions using the ranges given
    # continous conditions
    if label_col_contd is None:
        label_contd = None
    else:
        label_contd = torch.tensor(list(product(*[torch.arange(*lims)
                                                  for lims in label_contd_lims])))
    # discrete conditions
    if label_col_discr is None:
        label_discr = None
    else:
        label_discr = torch.tensor(list(product(*[torch.arange(*lims)
                                                  for lims in label_discr_lims])))
    # combine if both are given
    if label_contd is not None and label_discr is not None:
        # create the new tensor
        expanded = []
        for d in label_discr:
            # expand label_contd by appending each discrete condition to each row
            expanded_row = torch.cat((label_contd, d.expand(label_contd.shape[0], 1)), dim=1)


In [14]:
from train_cgans import wscGAN

ImportError: cannot import name 'wscGAN' from 'train_cgans' (/content/generative-mocap/train_cgans.py)

In [15]:
!tail -30 train_cgans.py

                         'knee_angle_r', 'ankle_angle_r'])
    # define col names
    value_cols = ['marker_gc', 'grf_3d_gc', 'ik_gc']
    names_cols = ['marker_names', 'grf_names_3d', 'ik_names']
    # labels in label_cols
    label_col_contd = ['age', 'mass', 'leglength_static', 'walking_speed']
    label_col_discr = ['gender_int']
    # file
    data_df_file = ['data/data_1.pickle', 'data/data_2.pickle']
    excluded_subjects = [2014001, 2014003, 2015042]
    # hyperparams
    z_dim = 20
    hidden_dim = 512
    n_epochs = 4000
    lr = 0.002
    batch_size = 128
    display_step = 250
    n_samples = 1
    # define ages at which n_samples of data will be generated
    label_contd_lims = [[15,71,10], [45,101,20],[785,1101,50],[0.16,2.41,0.5]]
    label_discr_lims = [[0, 1.01, 1]]
    model_name = 'multicgan'
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_cgan(marker_names, grf_names, ik_names, value_cols,
               names_cols, label_col_cont

In [16]:
!grep -n "def wscGAN" train_cgans.py

834:def wscGAN():


In [18]:
from train_cgans import train_cgan

In [19]:
from train_cgans import train_cgan
import torch
import numpy as np

In [20]:
n_epochs = 10
display_step = 1

In [22]:
import inspect
print(inspect.signature(train_cgan))

(marker_names=array(['L_IAS', 'L_IPS', 'R_IPS', 'R_IAS', 'R_FTC', 'R_FLE', 'R_FME',
       'R_FAX', 'R_TTC', 'R_FAL', 'R_TAM', 'R_FCC', 'R_FM1', 'R_FM2',
       'R_FM5'], dtype='<U5'), grf_names=array(['force', 'point', 'moment'], dtype='<U6'), ik_names=array(['pelvis_tilt', 'pelvis_list', 'pelvis_rotation', 'hip_flexion_r',
       'hip_adduction_r', 'hip_rotation_r', 'knee_angle_r',
       'ankle_angle_r'], dtype='<U15'), value_cols=['marker_gc', 'grf_3d_gc', 'ik_gc'], names_cols=['marker_names', 'grf_names_3d', 'ik_names'], label_col_contd=['age'], label_col_discr=None, data_df_file=['data/data_1.pickle', 'data/data_2.pickle'], excluded_subjects=[2014001, 2014003, 2015042], z_dim=20, hidden_dim=512, n_epochs=3000, lr=0.002, batch_size=128, display_step=250, n_samples=10, label_contd_lims=[[15, 71, 1]], label_discr_lims=None, model_name='acgan', device=device(type='cuda'), seed=0)


In [24]:
from train_cgans import train_cgan
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_cgan(
    label_col_contd=['walking_speed'],
    label_col_discr=None,
    n_epochs=10,
    display_step=1,
    label_contd_lims=[[0.16, 2.41, 0.02]],
    label_discr_lims=None,
    model_name='wscgan_test',
    device=device,
    seed=0
)

/content/generative-mocap/train_cgans.py:390: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  valid = Tensor(real.shape[0], 1).fill_(1.0)


Epoch 1, step 1: Generator loss: 0.06201788783073425, discriminator loss: 0.6747175455093384
Epoch 1, step 2: Generator loss: 0.04406166821718216, discriminator loss: 0.6202770471572876
Epoch 1, step 3: Generator loss: 0.032106149941682816, discriminator loss: 0.6538841724395752
Epoch 1, step 4: Generator loss: 0.020257655531167984, discriminator loss: 0.5602066516876221
Epoch 1, step 5: Generator loss: 0.015542843379080296, discriminator loss: 0.5498968362808228
Epoch 1, step 6: Generator loss: 0.013015050441026688, discriminator loss: 0.48784735798835754
Epoch 2, step 7: Generator loss: 0.00957079604268074, discriminator loss: 0.48393577337265015
Epoch 2, step 8: Generator loss: 0.008680985309183598, discriminator loss: 0.4262886345386505
Epoch 2, step 9: Generator loss: 0.007445447146892548, discriminator loss: 0.4446059465408325
Epoch 2, step 10: Generator loss: 0.006945331580936909, discriminator loss: 0.4376292824745178
Epoch 2, step 11: Generator loss: 0.006482699885964394, disc

In [25]:
!ls Results/wscgan_test

back_transform_input.pkl  synthetic_data_after_training.pkl
generator.pt		  transform_labels.pkl
